# TURFM Master Controller

This notebook is a self-contained controller for `hecras.py`.
It does not call `TURFM.main()`. Run the cells from top to bottom.

The shared step counter increments only after a step cell finishes successfully.


In [ ]:
from __future__ import annotations

import csv
from dataclasses import dataclass
import logging
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hecras import HECRAS

BUR_BUR_ROOT = PROJECT_ROOT / "1 Bur-Bur"
HEC_OUTPUT_ROOT = PROJECT_ROOT / "3 Hecras" / "code_generated"
HYDROLOGY_KMZ = PROJECT_ROOT / "2 Hydrology" / "Burdur Points.kmz"
PROJECTION_FILE = PROJECT_ROOT / "0 Proj" / "TUREF_CM30_projection.prj"
WORKING_FIXED_SHP_ROOT = PROJECT_ROOT / "working" / "fixed_shp"
SELF_EXAMPLE_ROOT = PROJECT_ROOT / "3 Hecras" / "self_example"
RAS_EXE_PATH = Path(r"C:\Program Files (x86)\HEC\HEC-RAS\6.6\Ras.exe")

CENTERLINE_SAMPLES_PER_SEGMENT = 500
HYDROLOGY_BUFFER_METERS = 150.0
BANK_STATION_MODE = "snap"
RIVER_LINE_METHOD = "simple_distance"

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
)
logger = logging.getLogger("TURFM_notebook")

STEP_STATE = {
    "completed_steps": 0,
    "history": [],
}


def complete_step(step_name: str, details: object | None = None) -> None:
    STEP_STATE["completed_steps"] += 1
    STEP_STATE["history"].append({
        "step": STEP_STATE["completed_steps"],
        "name": step_name,
        "details": details,
    })
    print(f"Step {STEP_STATE['completed_steps']} complete: {step_name}")
    if details is not None:
        print(details)


def show_progress() -> None:
    print(f"Completed steps: {STEP_STATE['completed_steps']}")
    for item in STEP_STATE["history"]:
        print(f"  {item['step']}. {item['name']}")


@dataclass(frozen=True)
class ModelConfig:
    model_folder: str
    model_dir: Path
    project_stem: str
    project_title: str
    hec_path: Path
    cross_section_csv: Path
    bank_lines_shp: Path
    structure_csv: Path | None


def available_model_dirs() -> list[Path]:
    return sorted(path for path in BUR_BUR_ROOT.iterdir() if path.is_dir())


def resolve_model_dir(model_folder: str) -> Path:
    direct_path = Path(model_folder)
    if direct_path.is_dir():
        return direct_path.resolve()

    candidate = BUR_BUR_ROOT / model_folder
    if candidate.is_dir():
        return candidate

    matches = [
        path
        for path in available_model_dirs()
        if path.name.casefold() == model_folder.casefold()
    ]
    if len(matches) == 1:
        return matches[0]

    available = ", ".join(path.name for path in available_model_dirs())
    raise FileNotFoundError(
        f"Model folder '{model_folder}' was not found under {BUR_BUR_ROOT}. "
        f"Available folders: {available}"
    )


def select_single_file(folder: Path, pattern: str, description: str) -> Path:
    candidates = sorted(folder.glob(pattern))
    if not candidates:
        raise FileNotFoundError(
            f"No {description} matching '{pattern}' was found in {folder}."
        )
    if len(candidates) > 1:
        names = ", ".join(path.name for path in candidates)
        raise FileNotFoundError(
            f"Expected one {description} in {folder}, found {len(candidates)}: "
            f"{names}"
        )
    return candidates[0]


def select_optional_single_file(folder: Path, pattern: str) -> Path | None:
    if not folder.is_dir():
        return None
    candidates = sorted(folder.glob(pattern))
    if not candidates:
        return None
    if len(candidates) > 1:
        names = ", ".join(path.name for path in candidates)
        raise FileNotFoundError(
            f"Expected at most one file matching '{pattern}' in {folder}, "
            f"found {len(candidates)}: {names}"
        )
    return candidates[0]


def build_model_config(model_folder: str) -> ModelConfig:
    model_dir = resolve_model_dir(model_folder)
    cross_section_csv = select_single_file(
        model_dir / "KESIT_TESLIM",
        "*.csv",
        "cross section CSV",
    )
    bank_lines_shp = select_single_file(
        model_dir / "SEV_USTU",
        "*.shp",
        "bank line shapefile",
    )
    structure_csv = select_optional_single_file(
        model_dir / "ROLEVE" / "structure_dim",
        "*.csv",
    )
    return ModelConfig(
        model_folder=model_dir.name,
        model_dir=model_dir,
        project_stem=model_dir.name,
        project_title=model_dir.name,
        hec_path=HEC_OUTPUT_ROOT / model_dir.name,
        cross_section_csv=cross_section_csv,
        bank_lines_shp=bank_lines_shp,
        structure_csv=structure_csv,
    )


def combined_junction_shp(
    main_model: ModelConfig,
    tributary_model: ModelConfig,
) -> Path | None:
    combined_name = (
        f"{main_model.model_folder.replace('BUR-BUR-MER-', '')}"
        f"__{tributary_model.model_folder.replace('BUR-BUR-MER-', '')}_combined.shp"
    )
    candidate = WORKING_FIXED_SHP_ROOT / combined_name
    if candidate.exists():
        return candidate
    fallback = (
        WORKING_FIXED_SHP_ROOT
        / "BUR-BUR-MER-ATATURK-Rev-V1__ATATURK-T_combined.shp"
    )
    if fallback.exists():
        return fallback
    return None


def junction_reference_geometry(main_model: ModelConfig) -> Path | None:
    candidate = SELF_EXAMPLE_ROOT / f"{main_model.project_stem}.g01"
    if candidate.exists():
        return candidate
    matches = sorted(SELF_EXAMPLE_ROOT.glob("*.g01"))
    if matches:
        return matches[0]
    return None


def junction_reference_project_name(
    reference_geometry: Path | None,
    fallback_name: str,
) -> str:
    if reference_geometry is None:
        return fallback_name
    return reference_geometry.stem or fallback_name


def read_river_name_from_csv(cross_section_csv: Path) -> str:
    with cross_section_csv.open(newline="", encoding="utf-8", errors="ignore") as handle:
        reader = csv.DictReader(handle)
        first_row = next(reader, None)
    if first_row is None:
        raise ValueError(f"No rows were found in {cross_section_csv}.")
    river_name = str(first_row.get("River", "")).strip()
    if not river_name:
        raise ValueError(f"Column 'River' was empty in {cross_section_csv}.")
    return river_name


def infer_junction_output_name(
    main_model: ModelConfig,
    tributary_model: ModelConfig,
    fallback_name: str,
) -> str:
    try:
        main_river = read_river_name_from_csv(main_model.cross_section_csv)
        tributary_river = read_river_name_from_csv(tributary_model.cross_section_csv)
        inferred = HECRAS.infer_junction_project_stem(
            main_river=main_river,
            tributary_river=tributary_river,
        )
        return inferred or fallback_name
    except Exception:
        return fallback_name


print(f"Notebook root: {PROJECT_ROOT}")
show_progress()


## Step 1: Configure The Run

Edit this cell before executing the workflow cells below.

In [ ]:
RUN_MODE = "junction"  # "single", "junction", or "list_models"

SINGLE_MODEL_FOLDER = "BUR-BUR-MER-ATATURK-Rev-V1"
MAIN_MODEL_FOLDER = "BUR-BUR-MER-ATATURK-Rev-V1"
TRIBUTARY_MODEL_FOLDER = "BUR-BUR-MER-ATATURK-T"

USE_STRUCTURES = True
SINGLE_STRUCTURE_CSV = None
MAIN_STRUCTURE_CSV = None
TRIBUTARY_STRUCTURE_CSV = None

RUN_CONFIG = {
    "run_mode": RUN_MODE,
    "single_model_folder": SINGLE_MODEL_FOLDER,
    "main_model_folder": MAIN_MODEL_FOLDER,
    "tributary_model_folder": TRIBUTARY_MODEL_FOLDER,
    "use_structures": USE_STRUCTURES,
    "single_structure_csv": SINGLE_STRUCTURE_CSV,
    "main_structure_csv": MAIN_STRUCTURE_CSV,
    "tributary_structure_csv": TRIBUTARY_STRUCTURE_CSV,
}

RUN_CONFIG


## Step 2: Validate Configuration And Project Paths

In [ ]:
required_paths = {
    "BUR_BUR_ROOT": BUR_BUR_ROOT,
    "HYDROLOGY_KMZ": HYDROLOGY_KMZ,
    "PROJECTION_FILE": PROJECTION_FILE,
    "HEC_OUTPUT_ROOT": HEC_OUTPUT_ROOT,
}

missing_paths = [name for name, path in required_paths.items() if not path.exists()]
if missing_paths:
    raise FileNotFoundError(f"Missing required project paths: {missing_paths}")

if RUN_MODE not in {"single", "junction", "list_models"}:
    raise ValueError("RUN_MODE must be 'single', 'junction', or 'list_models'.")

if RUN_MODE == "single" and not SINGLE_MODEL_FOLDER:
    raise ValueError("Set SINGLE_MODEL_FOLDER for single-model mode.")

if RUN_MODE == "junction" and (not MAIN_MODEL_FOLDER or not TRIBUTARY_MODEL_FOLDER):
    raise ValueError(
        "Set MAIN_MODEL_FOLDER and TRIBUTARY_MODEL_FOLDER for junction mode."
    )

available_models = [path.name for path in available_model_dirs()]
context = {
    "run_mode": RUN_MODE,
    "project_root": str(PROJECT_ROOT),
    "available_model_count": len(available_models),
}
complete_step("validated configuration", context)
context


## Step 3: Resolve Model Inputs

In [ ]:
NOTEBOOK_CONTEXT = {
    "run_mode": RUN_MODE,
    "available_models": [path.name for path in available_model_dirs()],
}

if RUN_MODE == "list_models":
    NOTEBOOK_CONTEXT["result"] = {"available_models": NOTEBOOK_CONTEXT["available_models"]}
elif RUN_MODE == "single":
    model = build_model_config(SINGLE_MODEL_FOLDER)
    NOTEBOOK_CONTEXT["model"] = model
    NOTEBOOK_CONTEXT["result"] = None
else:
    main_model = build_model_config(MAIN_MODEL_FOLDER)
    tributary_model = build_model_config(TRIBUTARY_MODEL_FOLDER)
    NOTEBOOK_CONTEXT["main_model"] = main_model
    NOTEBOOK_CONTEXT["tributary_model"] = tributary_model
    NOTEBOOK_CONTEXT["result"] = None

complete_step("resolved model inputs", NOTEBOOK_CONTEXT["run_mode"])
NOTEBOOK_CONTEXT


## Step 4: Prepare Runtime Objects And Output Targets

In [ ]:
hec = HECRAS(ras_exe_path=RAS_EXE_PATH)
NOTEBOOK_CONTEXT["hec"] = hec

if RUN_MODE == "single":
    model = NOTEBOOK_CONTEXT["model"]
    structure_csv = None
    if USE_STRUCTURES:
        structure_csv = (
            Path(SINGLE_STRUCTURE_CSV)
            if SINGLE_STRUCTURE_CSV
            else model.structure_csv
        )
    NOTEBOOK_CONTEXT["structure_csv"] = structure_csv
    NOTEBOOK_CONTEXT["output_folder"] = model.hec_path
elif RUN_MODE == "junction":
    main_model = NOTEBOOK_CONTEXT["main_model"]
    tributary_model = NOTEBOOK_CONTEXT["tributary_model"]
    main_structure_csv = None
    tributary_structure_csv = None
    if USE_STRUCTURES:
        main_structure_csv = (
            Path(MAIN_STRUCTURE_CSV)
            if MAIN_STRUCTURE_CSV
            else main_model.structure_csv
        )
        tributary_structure_csv = (
            Path(TRIBUTARY_STRUCTURE_CSV)
            if TRIBUTARY_STRUCTURE_CSV
            else tributary_model.structure_csv
        )
    combined_name = f"{main_model.model_folder}__{tributary_model.model_folder}"
    combined_bank_shp = combined_junction_shp(main_model, tributary_model)
    reference_geometry = junction_reference_geometry(main_model)
    output_name = junction_reference_project_name(
        reference_geometry,
        infer_junction_output_name(main_model, tributary_model, combined_name),
    )
    NOTEBOOK_CONTEXT["main_structure_csv"] = main_structure_csv
    NOTEBOOK_CONTEXT["tributary_structure_csv"] = tributary_structure_csv
    NOTEBOOK_CONTEXT["combined_bank_shp"] = combined_bank_shp
    NOTEBOOK_CONTEXT["reference_geometry"] = reference_geometry
    NOTEBOOK_CONTEXT["output_name"] = output_name
    NOTEBOOK_CONTEXT["output_folder"] = HEC_OUTPUT_ROOT / output_name

complete_step("prepared runtime objects", str(NOTEBOOK_CONTEXT.get("output_folder")))
{
    key: value
    for key, value in NOTEBOOK_CONTEXT.items()
    if key != "hec"
}


## Step 5: Assemble `hecras.py` Call Arguments

In [ ]:
if RUN_MODE == "list_models":
    NOTEBOOK_CONTEXT["call"] = {
        "method_name": None,
        "kwargs": None,
    }
elif RUN_MODE == "single":
    model = NOTEBOOK_CONTEXT["model"]
    NOTEBOOK_CONTEXT["call"] = {
        "method_name": "screen_steady_flows_from_kmz",
        "kwargs": {
            "project_folder": NOTEBOOK_CONTEXT["output_folder"],
            "project_stem": model.project_stem,
            "project_title": model.project_title,
            "cross_section_csv": model.cross_section_csv,
            "bank_lines_shp": model.bank_lines_shp,
            "structure_csv": NOTEBOOK_CONTEXT["structure_csv"],
            "hydrology_kmz": HYDROLOGY_KMZ,
            "buffer_distance": HYDROLOGY_BUFFER_METERS,
            "projection_file": PROJECTION_FILE,
            "centerline_samples_per_segment": CENTERLINE_SAMPLES_PER_SEGMENT,
            "bank_station_mode": BANK_STATION_MODE,
            "river_line_method": RIVER_LINE_METHOD,
        },
    }
else:
    main_model = NOTEBOOK_CONTEXT["main_model"]
    tributary_model = NOTEBOOK_CONTEXT["tributary_model"]
    NOTEBOOK_CONTEXT["call"] = {
        "method_name": "screen_steady_junction_flows_from_kmz",
        "kwargs": {
            "project_folder": NOTEBOOK_CONTEXT["output_folder"],
            "project_stem": NOTEBOOK_CONTEXT["output_name"],
            "project_title": NOTEBOOK_CONTEXT["output_name"],
            "main_cross_section_csv": main_model.cross_section_csv,
            "main_bank_lines_shp": main_model.bank_lines_shp,
            "tributary_cross_section_csv": tributary_model.cross_section_csv,
            "tributary_bank_lines_shp": tributary_model.bank_lines_shp,
            "main_structure_csv": NOTEBOOK_CONTEXT["main_structure_csv"],
            "tributary_structure_csv": NOTEBOOK_CONTEXT["tributary_structure_csv"],
            "combined_bank_lines_shp": NOTEBOOK_CONTEXT["combined_bank_shp"],
            "hydrology_kmz": HYDROLOGY_KMZ,
            "buffer_distance": HYDROLOGY_BUFFER_METERS,
            "reference_geometry_file": NOTEBOOK_CONTEXT["reference_geometry"],
            "projection_file": PROJECTION_FILE,
            "centerline_samples_per_segment": CENTERLINE_SAMPLES_PER_SEGMENT,
            "bank_station_mode": BANK_STATION_MODE,
            "river_line_method": RIVER_LINE_METHOD,
        },
    }

complete_step("assembled hecras call arguments", NOTEBOOK_CONTEXT["call"]["method_name"])
NOTEBOOK_CONTEXT["call"]


## Step 6: Execute The Workflow

In [ ]:
if RUN_MODE == "list_models":
    result = NOTEBOOK_CONTEXT["result"]
else:
    call = NOTEBOOK_CONTEXT["call"]
    screening = getattr(NOTEBOOK_CONTEXT["hec"], call["method_name"])(**call["kwargs"])
    NOTEBOOK_CONTEXT["screening"] = screening
    result = screening.to_dict()
    NOTEBOOK_CONTEXT["result"] = result

complete_step("executed workflow", RUN_MODE)
result


## Step 7: Review Outputs

In [ ]:
if RUN_MODE == "list_models":
    print("Available Bur-Bur model folders:")
    for name in NOTEBOOK_CONTEXT["result"]["available_models"]:
        print(name)
else:
    screening = NOTEBOOK_CONTEXT["screening"]
    print("=== Screening Results ===")
    print(screening.message)
    print(f"Report CSV: {screening.report_csv}")
    print(f"Report TXT: {screening.report_txt}")
    print(f"Final model return period: {screening.final_model_return_period}")
    print(f"Final model discharge (cms): {screening.final_model_flow_cms}")
    print(f"Maximum safe return period: {screening.max_safe_return_period}")
    print(f"Maximum safe discharge (cms): {screening.max_safe_flow_cms}")

complete_step("reviewed outputs")
show_progress()
NOTEBOOK_CONTEXT["result"]
